# 01 - Bronze: Ingestão dos dados de benefícios concedidos pelo INSS

## Objetivo

Realizar a ingestão de todos os arquivos CSV brutos para a camada Bronze do projeto.

A camada Bronze tem como princípios:

- preservar os dados exatamente como foram recebidos da fonte;
- não aplicar transformações de negócio;
- adicionar apenas metadados técnicos de rastreabilidade;
- garantir que qualquer análise futura possa ser rastreada até o arquivo de origem.

A tabela gerada nesta etapa é:

`afastamento_inss.bronze.beneficios_concedidos`

## 1. Importações

Nesta etapa são importadas as funções necessárias para a carga Bronze.

- `lit` — adiciona colunas com valor constante (metadado);
- `current_timestamp` — registra o momento exato da ingestão;
- `regexp_extract` — extrai o nome do arquivo e a competência do caminho;
- `col` — referência a colunas do DataFrame.

A rastreabilidade do arquivo de origem é obtida via `_metadata.file_path`, coluna especial disponibilizada pelo Spark em leituras de arquivos.

In [0]:
from pyspark.sql.functions import (
    lit,
    current_timestamp,
    regexp_extract,
    regexp_replace,
    col,
    count,
    when,
    concat_ws,
    substring
)


## 2. Parâmetros da carga

Os parâmetros centralizam as configurações da ingestão em um único lugar.

Isso garante que qualquer alteração futura, como a ingestão de uma nova competência, seja feita apenas aqui, sem precisar modificar o restante do notebook.

- `DIRETORIO_RAW` — caminho do diretório com os arquivos CSV no Volume Bronze;
- `TABELA_DESTINO` — tabela Delta que será criada ou sobrescrita.

In [0]:
DIRETORIO_RAW  = "/Volumes/afastamento_inss/bronze/raw/"
TABELA_DESTINO = "afastamento_inss.bronze.beneficios_concedidos"

print(f"Diretório: {DIRETORIO_RAW}")
print(f"Destino  : {TABELA_DESTINO}")


## 3. Leitura do arquivo CSV bruto

Os arquivos são lidos sem inferência de tipos (`inferSchema=false`).

Essa decisão é intencional na camada Bronze:

- a inferência automática pode alterar códigos numéricos com zeros à esquerda;
- datas podem ser convertidas incorretamente dependendo do formato;
- a Bronze deve preservar o dado exatamente como veio da fonte.

Todos os campos são lidos como `string` nesta etapa.

A tipagem correta será aplicada na camada Silver.

As opções utilizadas são:

- `header=true` — primeira linha contém os nomes das colunas;
- `inferSchema=false` — todos os campos lidos como string;
- `sep=;` — delimitador identificado na exploração.

Todos os arquivos CSV do diretório são lidos de uma única vez com o padrão `beneficios_concedidos_*.csv`.

O schema é determinado pelo primeiro arquivo (27 colunas, Grupo A). Arquivos com menos colunas têm valores `null` nas colunas ausentes. A coluna `_metadata.file_path` registra o arquivo de origem de cada linha, permitindo rastreabilidade por competência.

In [0]:
# Lista os arquivos CSV para ler separadamente (esquemas diferentes)
arquivos_csv = sorted(f.path for f in dbutils.fs.ls(DIRETORIO_RAW) if f.name.endswith(".csv"))

def identificar_formato(colunas):
    """Identifica o layout pelo cabecalho, e nao apenas pela quantidade de colunas.

    Existem dois layouts diferentes com 23 colunas:
    - 202406: ordem alfabetica (APS;CID;Classificador PA;...) + coluna vazia no final
    - 202407-202412 e 202502-202505: ordem original, sem as 4 colunas finais
    Agrupar so pela quantidade de colunas desalinhava todos os arquivos do segundo grupo.
    """
    n = len(colunas)
    # Spark renomeia cabecalhos duplicados (ex.: APS0, APS1), por isso startswith
    if colunas[1].upper().startswith("CID"):
        return f"alfabetico_{n}"
    return f"original_{n}"

dfs_por_formato = {}
arquivos_por_formato = {}
for arquivo in arquivos_csv:
    df_file = (
        spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("sep", ";")
            .csv(arquivo)
    )
    formato = identificar_formato(df_file.columns)
    df_file = df_file.selectExpr("*", f"'{arquivo}' as _source_file")
    dfs_por_formato.setdefault(formato, []).append(df_file)
    arquivos_por_formato.setdefault(formato, []).append(arquivo.split("/")[-1])

print(f"Arquivos: {len(arquivos_csv)}")
for formato, nomes in sorted(arquivos_por_formato.items()):
    print(f"  {formato:<15} {len(nomes):>3} arquivo(s): {', '.join(nomes)}")


## 4. Padronização dos nomes das colunas e adição de metadados

O Delta Lake não aceita caracteres especiais nos nomes das colunas, incluindo espaços, acentos e pontos.

Os nomes originais identificados na exploração continham:

- espaços entre palavras: `Mun Resid`, `Ramo Atividade`;
- acentos: `Competência concessão`, `Vínculo dependentes`;
- ponto no final: `Sexo.`;
- espaços no início e no final: ` Qt SM RMI `;
- pontos no meio: `CNAE 2.023`, `CNAE 2.024`.

A padronização adota o padrão `snake_case`:

- letras minúsculas;
- palavras separadas por `_`;
- sem acentos;
- sem caracteres especiais.

A função `toDF()` renomeia todas as colunas pela posição de uma só vez, sem depender dos nomes originais com caracteres especiais.

Como os arquivos possuem esquemas diferentes, o formato de cada arquivo é identificado pelo **cabeçalho**, e não apenas pela quantidade de colunas: existem dois layouts distintos com 23 colunas (`202406` em ordem alfabética e `202407–202412`/`202502–202505` na ordem original sem as 4 colunas finais). Cada formato tem seu próprio mapeamento posicional e colunas ausentes recebem `null`. No layout alfabético, `aps_cod` e `cid_cod` são extraídos das respectivas descrições. A competência e o nome do arquivo são extraídos do caminho via `_metadata.file_path`.

Após a renomeação são adicionados três metadados técnicos:

- `_arquivo_origem` — nome do arquivo CSV de origem;
- `_competencia` — competência de referência do arquivo;
- `_data_ingestao` — timestamp de quando a carga foi executada.

O prefixo `_` distingue os metadados técnicos das colunas de negócio.

In [0]:
# Colunas na ordem original completa (27 colunas)
COLUNAS_PADRAO = [
    "aps_cod", "aps_desc", "competencia_concessao", "especie_cod", "especie_desc",
    "cid_cod", "cid_desc", "despacho_cod", "despacho_desc", "dt_nascimento",
    "sexo", "clientela", "mun_resid", "vinculo_dependentes", "forma_filiacao",
    "uf", "qt_sm_rmi", "ramo_atividade", "dt_dcb", "dt_ddb", "dt_dib",
    "pais_acordo_internacional", "classificador_pa", "cnae_2023", "cnae_2024",
    "grau_instrucao", "qt_anos_contribuicao"
]

# Mapeamento posicional por formato (identificado pelo cabecalho). _source_file vem ao final.
# original_27   (202306-202405, 202511-202607): formato original completo
# original_26   (202506-202510): sem Qt Anos Contribuicao
# original_23   (202407-202412, 202502-202505): sem CNAE 2.0 (x2), Grau Instrucao e Qt Anos Contribuicao
# original_22   (202501): como original_23, mas o Despacho vem so com a descricao
# alfabetico_23 (202406): ordem alfabetica, sem codigos de APS/CID, com Segurado MEI e coluna vazia
MAPEAMENTO_POR_FORMATO = {
    "original_27": COLUNAS_PADRAO + ["_source_file"],
    "original_26": COLUNAS_PADRAO[:26] + ["_source_file"],
    "original_23": COLUNAS_PADRAO[:23] + ["_source_file"],
    "original_22": [
        "aps_cod", "aps_desc", "competencia_concessao", "especie_cod", "especie_desc",
        "cid_cod", "cid_desc", "despacho_desc", "dt_nascimento", "sexo",
        "clientela", "mun_resid", "vinculo_dependentes", "forma_filiacao", "uf",
        "qt_sm_rmi", "ramo_atividade", "dt_dcb", "dt_ddb", "dt_dib",
        "pais_acordo_internacional", "classificador_pa", "_source_file"
    ],
    "alfabetico_23": [
        "aps_desc", "cid_desc", "classificador_pa", "clientela", "competencia_concessao",
        "despacho_cod", "despacho_desc", "especie_cod", "especie_desc", "forma_filiacao",
        "mun_resid", "ramo_atividade", "segurado_mei", "sexo", "uf",
        "vinculo_dependentes", "pais_acordo_internacional", "qt_sm_rmi", "dt_dcb",
        "dt_ddb", "dt_dib", "dt_nascimento", "_descartar", "_source_file"
    ],
}

COLUNAS_EXTRAS = ["segurado_mei", "_descartar"]

partes = []
for formato, group_dfs in dfs_por_formato.items():
    if formato not in MAPEAMENTO_POR_FORMATO:
        raise ValueError(f"Formato desconhecido: {formato} ({arquivos_por_formato[formato]})")

    nomes = MAPEAMENTO_POR_FORMATO[formato]
    # Renomeia cada arquivo antes do union, garantindo alinhamento por nome
    dfs_renomeados = [df.toDF(*nomes) for df in group_dfs]
    df_group = dfs_renomeados[0]
    for df in dfs_renomeados[1:]:
        df_group = df_group.unionByName(df)

    # Remover colunas extras (Segurado MEI, coluna vazia final)
    for c in COLUNAS_EXTRAS:
        if c in df_group.columns:
            df_group = df_group.drop(c)

    # O layout alfabetico traz apenas as descricoes de APS e CID
    # (ex.: "02001050-Aps Maceio..." e "F72   Retardo Mental Grave").
    # Os codigos sao extraidos da descricao para seguir o padrao dos demais arquivos.
    if formato.startswith("alfabetico"):
        df_group = (
            df_group
            .withColumn("aps_cod", regexp_extract(col("aps_desc"), r"^(\d+)-", 1))
            .withColumn(
                "cid_cod",
                when(
                    col("cid_desc").rlike(r"^[A-Za-z]\d"),
                    regexp_replace(regexp_extract(col("cid_desc"), r"^(\S+)", 1), r"\.", "")
                ).otherwise(col("cid_desc"))
            )
        )

    # Adicionar colunas ausentes como null
    for c in COLUNAS_PADRAO:
        if c not in df_group.columns:
            df_group = df_group.withColumn(c, lit(None).cast("string"))

    # Selecionar em ordem padrao + _source_file
    df_group = df_group.select(*COLUNAS_PADRAO, "_source_file")
    partes.append(df_group)

df_bronze = partes[0]
for p in partes[1:]:
    df_bronze = df_bronze.unionByName(p)

df_bronze = (
    df_bronze
    .withColumn("_arquivo_origem", regexp_extract(col("_source_file"), r'([^/]+\.csv)$', 1))
    .withColumn("_competencia",    regexp_extract(col("_source_file"), r'(\d{6})', 1))
    .withColumn("_data_ingestao",  current_timestamp())
    .drop("_source_file")
)

print(f"Colunas com metadados: {len(df_bronze.columns)}")
print(df_bronze.columns)


## 4.1 Checagem de alinhamento das colunas (antes de gravar)

Antes de gravar a Bronze, cada arquivo é verificado para garantir que o mapeamento posicional está correto:

- `competencia_concessao` deve corresponder à competência do nome do arquivo (`yyyyMM` ou `yyyy-MM-01 ...`);
- `especie_cod` deve ser numérico;
- `sexo` deve ser `Masculino`, `Feminino` ou sentinela.

Se algum arquivo tiver menos de 99% das linhas válidas, a execução é interrompida e a tabela **não** é gravada. Isso impede que colunas desalinhadas cheguem às camadas Silver e Gold.

In [0]:
from pyspark.sql.functions import sum as spark_sum

LIMIAR_ALINHAMENTO = 0.99

competencia_ok = (
    (col("competencia_concessao") == col("_competencia"))
    | col("competencia_concessao").startswith(
        concat_ws("-", substring(col("_competencia"), 1, 4), substring(col("_competencia"), 5, 2))
    )
)
especie_ok = col("especie_cod").rlike(r"^\d{1,3}$")
sexo_ok    = col("sexo").isNull() | col("sexo").isin("Masculino", "Feminino", "Ignorado", "{ñ class}", "")

df_checagem = (
    df_bronze
    .groupBy("_arquivo_origem")
    .agg(
        count("*").alias("linhas"),
        spark_sum(when(competencia_ok, 1).otherwise(0)).alias("competencia_ok"),
        spark_sum(when(especie_ok, 1).otherwise(0)).alias("especie_ok"),
        spark_sum(when(sexo_ok, 1).otherwise(0)).alias("sexo_ok"),
    )
    .withColumn("pct_competencia", col("competencia_ok") / col("linhas"))
    .withColumn("pct_especie",     col("especie_ok") / col("linhas"))
    .withColumn("pct_sexo",        col("sexo_ok") / col("linhas"))
    .orderBy("_arquivo_origem")
    .cache()
)

display(df_checagem)

problemas = df_checagem.filter(
    (col("pct_competencia") < LIMIAR_ALINHAMENTO)
    | (col("pct_especie") < LIMIAR_ALINHAMENTO)
    | (col("pct_sexo") < LIMIAR_ALINHAMENTO)
).collect()

if problemas:
    for r in problemas:
        print(f"  {r['_arquivo_origem']}: competencia={r['pct_competencia']:.1%} "
              f"especie={r['pct_especie']:.1%} sexo={r['pct_sexo']:.1%}")
    raise ValueError(f"{len(problemas)} arquivo(s) com colunas desalinhadas. Bronze NAO foi gravada.")

print(f"Alinhamento OK em todos os {df_checagem.count()} arquivos.")


## 5. Gravação da tabela Delta Bronze

O DataFrame é gravado como tabela Delta no schema `bronze` do catálogo `afastamento_inss`, contendo todos os arquivos ingeridos.

As opções utilizadas são:

- `format=delta` — formato Delta Lake, que garante transações ACID e versionamento;
- `mode=overwrite` — substitui a tabela caso já exista, permitindo reprocessamento;
- `overwriteSchema=true` — atualiza o schema da tabela caso tenha mudado.

A combinação de `overwrite` com `overwriteSchema=true` é adequada para a Bronze porque:

- o arquivo de origem pode mudar entre execuções;
- não existe transformação acumulativa nesta camada;
- a Bronze pode ser recriada integralmente a partir do arquivo bruto a qualquer momento.

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")
print(f"Total de linhas: {df_bronze.count():,}")


## 6. Validação da carga

A validação compara a contagem de linhas e colunas entre o arquivo bruto e a tabela Bronze gravada.

Essa etapa garante que:

- nenhum registro foi perdido durante a renomeação ou gravação;
- a quantidade de colunas de negócio é idêntica à do arquivo de origem;
- os três metadados técnicos foram adicionados corretamente.

O padrão de validação adotado neste projeto é:

> Toda camada valida contra a camada anterior.

- Raw → Bronze (todos os arquivos)
- Bronze → Silver
- Silver → Gold

In [0]:
# Contagens totais
linhas_raw     = df_bronze.count()
linhas_bronze  = spark.table(TABELA_DESTINO).count()
colunas_bronze = len(spark.table(TABELA_DESTINO).columns)

# Comparação
print("=" * 55)
print("VALIDAÇÃO DA CARGA BRONZE — INGESTÃO MULTI-ARQUIVO")
print("=" * 55)
print(f"{'Total de linhas (raw)':<35} {linhas_raw:>10,}")
print(f"{'Total de linhas (bronze)':<35} {linhas_bronze:>10,}")
print(f"{'Colunas (originais + metadados)':<35} {colunas_bronze}")
print("-" * 55)

# Distribuição por competência
print(f"\nDistribuição por competência:")
spark.table(TABELA_DESTINO) \
    .groupBy("_competencia") \
    .count() \
    .orderBy("_competencia") \
    .show(40, truncate=False)

# Verificação de integridade
if linhas_raw == linhas_bronze:
    print("Linhas: OK — nenhum registro perdido")
else:
    diff = linhas_raw - linhas_bronze
    print(f"Linhas: DIVERGÊNCIA de {diff:,} registros")

colunas_esperadas = 27 + 3  # 27 originais + 3 metadados
if colunas_bronze == colunas_esperadas:
    print(f"Colunas: OK — {colunas_esperadas} (27 originais + 3 metadados)")
else:
    print(f"Colunas: DIVERGÊNCIA — esperado {colunas_esperadas}, encontrado {colunas_bronze}")

print("=" * 55)

# Amostra
display(spark.table(TABELA_DESTINO).limit(10))


## Conclusão

A tabela `afastamento_inss.bronze.beneficios_concedidos` foi criada com sucesso.

| Item | Valor |
|---|---|
| Arquivo de origem | 38 CSVs (202306–202607) |
| Competência | 202306–202607 (38 arquivos) |
| Linhas ingeridas | ver validação |
| Colunas originais | 27 |
| Colunas com metadados | 30 |
| Formato | Delta |

### Colunas de metadados adicionadas

| Coluna | Descrição |
|---|---|
| `_arquivo_origem` | Nome do arquivo CSV de origem |
| `_competencia` | Competência de referência no formato AAAAMM |
| `_data_ingestao` | Timestamp de execução da carga |

### Decisões técnicas desta etapa

| Decisão | Justificativa |
|---|---|
| `inferSchema=false` | Evita conversão incorreta de códigos e datas |
| `toDF()` para renomear | Evita dependência de nomes com caracteres especiais |
| `mode=overwrite` | Permite reprocessamento integral da Bronze |
| Prefixo `_` nos metadados | Distingue metadados técnicos das colunas de negócio |

### Próxima etapa

`02_silver_transformacao`

Aplicar regras de qualidade, tipagem, padronização e classificação de CID e espécie de benefício com base nos achados do notebook `00_exploracao`.

### Atenção: esquemas divergentes

Os arquivos CSV possuem 5 layouts diferentes (`original_27`, `original_26`, `original_23`, `original_22` e `alfabetico_23`). O layout é identificado pelo cabeçalho de cada arquivo, e a checagem da seção 4.1 interrompe a execução se algum arquivo ficar desalinhado.